# 02_silver_transformation

Clean and validate retail order data from the Bronze layer before storing it in a Silver Delta table.

## Table of Contents
1. Import PySpark and load Bronze data
2. Remove duplicates and empty rows
3. Clean text fields and handle missing store names
4. Convert and validate dates
5. Create Silver catalog and schema
6. Save and validate the Silver Delta table
7. Validate Silver data quality

### Import PySpark and load Bronze data

In [0]:
from pyspark.sql import functions as F

# 1. Hent data fra Bronze
df = spark.table(
    "dataengineering_bronze.raw_data.butiksordrer"
)

### Remove duplicates and empty rows

In [0]:
# 2. Fjern dubletter
df = df.dropDuplicates()

# 3. Fjern rækker, hvor alle værdier er NULL
df = df.na.drop(how="all")

### Clean text fields and handle missing store names

In [0]:
# 4. Fjern mellemrum fra tekstfelter
for col_name, dtype in df.dtypes:
    if dtype == "string":
        df = df.withColumn(
            col_name,
            F.trim(F.col(col_name))
        )

# 5. Konverter tomme tekstfelter til NULL
for col_name, dtype in df.dtypes:
    if dtype == "string":
        df = df.withColumn(
            col_name,
            F.when(
                F.col(col_name) == "",
                F.lit(None)
            ).otherwise(F.col(col_name))
        )

# 6. Erstat manglende butiksnavne med "Ukendt"
df = df.withColumn(
    "butik",
    F.when(
        F.col("butik").isNull(),
        F.lit("Ukendt")
    ).otherwise(F.col("butik"))
)

### Convert and validate dates

In [0]:
# 6. Konverter og valider datoer
df = df.withColumn(
    "ordre_dato",
    F.expr("try_cast(ordre_dato AS DATE)")
)

df = df.filter(
    F.col("ordre_dato").isNotNull()
)

### Create Silver catalog and schema

In [0]:
# 7. Opret Silver-katalog og schema
spark.sql("""
    CREATE CATALOG IF NOT EXISTS dataengineering_silver
""")

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    dataengineering_silver.cleaned_data
""")

### Save and validate the Silver Delta table

In [0]:
# 8. Gem som Delta-tabel
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "dataengineering_silver.cleaned_data.butiksordrer"
    )

# 9. Kontrollér den gemte tabel
df_check = spark.table(
    "dataengineering_silver.cleaned_data.butiksordrer"
)

display(df_check)

### Validate Silver data quality

In [0]:
from pyspark.sql import functions as F

# 7. Load the saved Silver table
df_check = spark.table(
    "dataengineering_silver.cleaned_data.butiksordrer"
)

# Total number of rows
print(f"Total rows: {df_check.count()}")

# Count NULL values in each column
null_counts = df_check.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in df_check.columns
])

print("NULL values per column:")
display(null_counts)

# Count duplicate rows
duplicate_count = (
    df_check.count()
    - df_check.dropDuplicates().count()
)

print(f"Duplicate rows: {duplicate_count}")

# Count orders with unknown store
unknown_stores = df_check.filter(
    F.col("butik") == "Ukendt"
).count()

print(f"Rows with unknown store: {unknown_stores}")